# Day 9: Stress Simulation
**Situation:** Simulate a normal day, a viral event, and an extreme spike across different strategies.

In [ ]:
class Message:
    def __init__(self, user_id, channel_id, content):
        self.user_id = user_id
        self.channel_id = channel_id
        self.content = content

class Shard:
    def __init__(self, shard_id):
        self.id = shard_id
        self.messages = []

    def store(self, message):
        self.messages.append(message)

class ShardManager:
    def __init__(self, num_shards):
        self.shards = [Shard(i) for i in range(num_shards)]

    def stats(self):
        for shard in self.shards:
            print(f"Shard {shard.id}: {len(shard.messages)} messages")

In [ ]:
# Redefining the managers for simulation
class ChannelShardManager(ShardManager):
    def send_message(self, message):
        self.shards[message.channel_id % len(self.shards)].store(message)

import hashlib
class HashShardManager(ShardManager):
    def send_message(self, message):
        # Simulating random message_id hashing
        message_key = f"{message.user_id}-{message.channel_id}-{random.random()}"
        h = int(hashlib.md5(str(message_key).encode()).hexdigest(), 16)
        self.shards[h % len(self.shards)].store(message)

In [ ]:
import random
def simulate(manager, num_users=1000, num_messages=5000):
    for _ in range(num_messages):
        user_id = random.randint(1, num_users)
        channel_id = random.randint(1, 50)
        msg = Message(user_id, channel_id, "hello")
        manager.send_message(msg)
        
    # Simulate Spike on one channel
    for _ in range(num_messages * 2):
        msg = Message(random.randint(1, num_users), 99, "Wow crazy game!")
        manager.send_message(msg)

    for shard in manager.shards:
        # Detect hotspot warning!
        percentage = (len(shard.messages) / (num_messages * 3)) * 100
        warning = " 🔥 HOTSPOT DETECTED!" if percentage > 50 else ""
        print(f"Shard {shard.id}: {len(shard.messages)} messages ({percentage:.1f}%){warning}")

### Running Simulation: Channel Sharding

In [ ]:
print("Testing ChannelShardManager")
manager_channel = ChannelShardManager(3)
simulate(manager_channel)

### Running Simulation: Hash Sharding

In [ ]:
print("Testing HashShardManager")
manager_hash = HashShardManager(3)
simulate(manager_hash)